# Rock-Paper-Scissors: MobileNetV2 (transfer learning)

**Purpose:** Transfer-learning model. MobileNetV2, pretrained on ImageNet (1.2M pictures),
is adapted to classify hand gestures from the OV7675 camera as rock, paper or scissors.
It tests whether pretrained features beat a CNN trained from scratch, and is compared
against the custom CNN in `Models/RESULTS.md`.
Based on [RPS: CNN vs MobileNet Transfer Learning](https://www.kaggle.com/code/adnanzamanniloy/rps-cnn-vs-mobilenet-transfer-learning)
(Kaggle), adapted for a grayscale camera, a separate test set and 4 classes.

**Model** (`build_mobilenet` in `model.py`)
- MobileNetV2 backbone with ImageNet weights, frozen at first
- New head: Dropout 0.3 → Linear 1280→128 → ReLU → Dropout 0.3 → Linear to 4 classes
- Fine-tuning: the last 2 backbone blocks are unfrozen partway through training
- About 2.4M parameters; about 164k trained before unfreezing

**Data**
- Kaggle Rock-Paper-Scissors dataset: 2,520 training, 33 validation, 372 test pictures
- Converted to grayscale, copied to 3 channels and resized to 224×224 (the size MobileNetV2 expects),
  normalized with ImageNet mean and standard deviation
- Same augmentation as the CNN: random crop, rotation (±30°), horizontal flip, brightness/contrast

**Changes from the Kaggle notebook**
- Always grayscale, since the camera has no color (the original trains mostly on color)
- Uses Kaggle's separate `test/` folder instead of a random split of `train/`, whose near-identical
  frames inflate accuracy
- Keeps one optimizer when unfreezing, so the learning-rate scheduler keeps working

**This notebook**
1. Loads the data and the pretrained MobileNetV2
2. Trains the new head, then fine-tunes the last backbone blocks
3. Evaluates on the test set: accuracy, confusion matrix, per-class precision and recall
4. Shows misclassified pictures
5. Saves the weights to `weights/mobilenet.pt`, the figures to `Models/figures/`, and updates `Models/RESULTS.md`

## Setup
On Google Colab, the first cell downloads the project and the data (pick **Runtime → Change runtime type → T4 GPU** first).
On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/carterjohndixon/EE-105-Project.git
    %cd EE-105-Project
    !wget -q https://github.com/carterjohndixon/EE-105-Project/releases/download/data-v1/data-zip.zip
    !unzip -q -o data-zip.zip && rm data-zip.zip

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

# find the project folder (the one with model.py), wherever this notebook is opened from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model.py").exists())
sys.path.insert(0, str(ROOT))
import model

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RETRAIN = False          # True: train again even if weights/ already has this model

KIND = "mobilenet"
NAME = "MobileNetV2"
BUILD = model.build_mobilenet
EPOCHS = 8
LR = 3e-4
UNFREEZE_AT = 4          # epoch at which the last 2 backbone blocks start training too

WEIGHTS = ROOT / "weights" / f"{KIND}.pt"
HISTORY = ROOT / "results" / f"history_{KIND}.json"
METRICS = ROOT / "results" / f"metrics_{KIND}.json"
FIGURES = ROOT / "Models" / "figures"
for d in (WEIGHTS.parent, HISTORY.parent, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.savefig(FIGURES / f"{KIND}_{name}.png", dpi=120, bbox_inches="tight")

print("project:", ROOT)
print("device:", DEVICE, "| torch", torch.__version__)

## Data

In [ ]:
for split in ("train", "validation", "test"):
    ds = datasets.ImageFolder(model.KAGGLE / split)
    counts = np.bincount(ds.targets, minlength=len(ds.classes))
    print(f"{split:<10}", dict(zip(ds.classes, counts.tolist())))

def show_tensor(ax, x):
    x = x.clone()
    if x.shape[0] == 3:                      # undo the ImageNet normalization
        x = x * torch.tensor(model.IMAGENET_STD)[:, None, None] + torch.tensor(model.IMAGENET_MEAN)[:, None, None]
    else:
        x = x * 0.5 + 0.5
    ax.imshow(x[0].clamp(0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_axis_off()

train_ds = model.make_dataset(KIND, "train")
test_ds = model.make_dataset(KIND, "test")
fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for i, ax in enumerate(axes[0]):
    x, y = train_ds[i * 400]
    show_tensor(ax, x); ax.set_title(f"train: {model.CLASSES[y]}", fontsize=10)
for i, ax in enumerate(axes[1]):
    x, y = test_ds[i * 60]
    show_tensor(ax, x); ax.set_title(f"test: {model.CLASSES[y]}", fontsize=10)
fig.suptitle(f"What the {NAME} sees: {tuple(x.shape[1:])} pixels, {x.shape[0]} channel(s)")
plt.tight_layout(); savefig("samples"); plt.show()

## Model

In [ ]:
net = BUILD()
n_total = sum(p.numel() for p in net.parameters())
n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(net if KIND == "cnn" else net.classifier)
print(f"\n{n_total:,} parameters, {n_train:,} trained at the start")

## Train
Keeps the weights from the best validation epoch. The validation set has only 33 pictures
(each one is 3%), so judge the model by the test set below, not by validation accuracy.
With `RETRAIN = False`, a model already saved in `weights/` is loaded instead.

In [ ]:
if WEIGHTS.exists() and HISTORY.exists() and not RETRAIN:
    net.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
    net.to(DEVICE)
    hist = json.loads(HISTORY.read_text())
    print(f"loaded {WEIGHTS.name} (set RETRAIN = True to train again)")
else:
    loaders = model.make_loaders(KIND)
    hist = model.train(net, loaders, epochs=EPOCHS, lr=LR, device=DEVICE,
                       unfreeze_at=UNFREEZE_AT, weights_path=WEIGHTS)
    HISTORY.write_text(json.dumps(hist))

In [ ]:
ep = np.arange(1, len(hist["train_acc"]) + 1)
best = int(np.argmax(hist["val_acc"])) + 1
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ep, hist["train_acc"], "--", label="train")
axes[0].plot(ep, hist["val_acc"], "-o", label="validation")
axes[1].plot(ep, hist["train_loss"], "--", label="train")
axes[1].plot(ep, hist["val_loss"], "-o", label="validation")
axes[0].set(title=f"{NAME}: accuracy", xlabel="epoch", ylim=(0, 1.02))
axes[1].set(title=f"{NAME}: loss", xlabel="epoch")
for ax in axes:
    ax.axvline(best, color="gray", lw=1, ls=":", label=f"kept (epoch {best})")
    if UNFREEZE_AT is not None:
        ax.axvline(UNFREEZE_AT + 0.5, color="tab:red", lw=1, ls="--", label="unfrozen")
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); savefig("training_curves"); plt.show()

## Test set

In [ ]:
loader = model.make_loaders(KIND)["test"]
t0 = time.perf_counter()
y_true, y_pred, probs = model.predict_loader(net, loader, DEVICE)
ms = 1000 * (time.perf_counter() - t0) / len(y_true)
acc = float((y_true == y_pred).mean())
print(f"{NAME}: {acc:.1%} test accuracy ({int((y_true != y_pred).sum())} wrong out of {len(y_true)}), "
      f"{ms:.1f} ms per picture including loading it from disk")

present = sorted(set(y_true.tolist()))                     # classes in the test set (no 'none')
cm = model.confusion(y_true, y_pred)[present]
fig, ax = plt.subplots(figsize=(6, 4.6))
ax.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(len(model.CLASSES)), model.CLASSES)
ax.set_yticks(range(len(present)), [model.CLASSES[c] for c in present])
ax.set(xlabel="predicted", ylabel="true", title=f"{NAME}: test accuracy {acc:.1%}")
plt.tight_layout(); savefig("confusion"); plt.show()

per_class = {}
print(f"\n{'class':<10}{'precision':>10}{'recall':>8}{'n':>6}")
for c in present:
    tp = int(np.sum((y_pred == c) & (y_true == c)))
    prec = tp / max(1, int(np.sum(y_pred == c)))
    rec = tp / max(1, int(np.sum(y_true == c)))
    per_class[model.CLASSES[c]] = {"precision": prec, "recall": rec}
    print(f"{model.CLASSES[c]:<10}{prec:>10.3f}{rec:>8.3f}{int(np.sum(y_true == c)):>6}")

METRICS.write_text(json.dumps({
    "name": NAME, "test_acc": acc, "wrong": int((y_true != y_pred).sum()), "n_test": len(y_true),
    "best_val_acc": max(hist["val_acc"]), "epochs": len(hist["train_acc"]),
    "seconds_per_epoch": float(np.mean(hist["seconds"])), "ms_per_image": ms,
    "params": n_total, "params_trained_at_start": n_train, "per_class": per_class,
    "device": DEVICE,
}, indent=1))

## Mistakes
Up to 8 wrong test predictions, with the confidence of the wrong answer.

In [ ]:
wrong = np.nonzero(y_true != y_pred)[0]
print(f"{len(wrong)} wrong out of {len(y_true)}")
if len(wrong):
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
    for ax in axes:
        ax.set_axis_off()
    for ax, i in zip(axes, wrong[:8]):
        show_tensor(ax, test_ds[i][0])
        p = y_pred[i]
        ax.set_title(f"true {model.CLASSES[y_true[i]]}\nsaid {model.CLASSES[p]} {probs[i, p]:.0%}",
                     fontsize=9, color="tab:red")
    savefig("mistakes"); plt.show()

## Our own pictures (`data/test`)
Kaggle's pictures are computer-rendered hands on white backgrounds. Pictures from our camera are the real check.
Skipped until `data/test` has pictures.

In [ ]:
own = ROOT / "data" / "test"
n_own = sum(1 for p in own.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg")) if own.exists() else 0
if n_own == 0:
    print("data/test has no pictures yet: capture some into data/test/<class>/ and run this cell again.")
else:
    ds = datasets.ImageFolder(own, transform=model.make_transform(KIND, train=False), allow_empty=True)
    ds.target_transform = model._ToClasses(ds.classes)
    yt, yp, _ = model.predict_loader(net, torch.utils.data.DataLoader(ds, batch_size=32), DEVICE)
    print(f"{NAME}: {(yt == yp).mean():.1%} on {len(yt)} of our pictures")
    print(model.confusion(yt, yp))

## Update `Models/RESULTS.md`
Rebuilds the comparison page from the saved results of **both** models (whichever have been run).
Running either notebook refreshes it. Write conclusions in `Models/NOTES.md` instead: this file is overwritten.

In [ ]:
def write_results_md():
    runs = {}
    for kind in ("cnn", "mobilenet"):
        m, h = ROOT / "results" / f"metrics_{kind}.json", ROOT / "results" / f"history_{kind}.json"
        if m.exists() and h.exists():
            runs[kind] = (json.loads(m.read_text()), json.loads(h.read_text()))

    if len(runs) == 2:                                   # side-by-side training curves
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        for (kind, (m, h)), color in zip(runs.items(), ("tab:blue", "tab:orange")):
            e = np.arange(1, len(h["val_acc"]) + 1)
            axes[0].plot(e, h["val_acc"], "-o", color=color, ms=3, label=f"{m['name']} validation")
            axes[0].plot(e, h["train_acc"], "--", color=color, label=f"{m['name']} train")
            axes[1].plot(e, h["val_loss"], "-o", color=color, ms=3, label=f"{m['name']} validation")
            axes[1].plot(e, h["train_loss"], "--", color=color, label=f"{m['name']} train")
        axes[0].set(title="Accuracy", xlabel="epoch", ylim=(0, 1.02)); axes[1].set(title="Loss", xlabel="epoch")
        for ax in axes:
            ax.grid(alpha=0.3); ax.legend(fontsize=9)
        plt.tight_layout()
        plt.savefig(FIGURES / "comparison_curves.png", dpi=120, bbox_inches="tight"); plt.close(fig)

    lines = ["# Model comparison: small CNN vs MobileNetV2", "",
             "Generated by the notebooks in `Models/`; do not edit by hand (it is overwritten).",
             "Test set: Kaggle `test/` folder, converted to grayscale like the camera.", "",
             "| Model | Test accuracy | Wrong | Best val | Parameters | Trained at start | Epochs | s/epoch | ms/picture |",
             "|---|---|---|---|---|---|---|---|---|"]
    for kind, (m, h) in runs.items():
        lines.append(f"| {m['name']} | **{m['test_acc']:.1%}** | {m['wrong']}/{m['n_test']} | {m['best_val_acc']:.1%} "
                     f"| {m['params']:,} | {m['params_trained_at_start']:,} | {m['epochs']} "
                     f"| {m['seconds_per_epoch']:.0f} ({m['device']}) | {m['ms_per_image']:.1f} |")
    lines += ["", "## Per class (test set)", "", "| Model | Class | Precision | Recall |", "|---|---|---|---|"]
    for kind, (m, h) in runs.items():
        for c, v in m["per_class"].items():
            lines.append(f"| {m['name']} | {c} | {v['precision']:.3f} | {v['recall']:.3f} |")
    if len(runs) == 2:
        lines += ["", "## Training curves, both models", "", "![Comparison](figures/comparison_curves.png)"]
    for kind, (m, h) in runs.items():
        lines += ["", f"## {m['name']}", "",
                  f"![{m['name']} training curves](figures/{kind}_training_curves.png)", "",
                  f"![{m['name']} confusion matrix](figures/{kind}_confusion.png)", "",
                  f"![{m['name']} mistakes](figures/{kind}_mistakes.png)"]
    conclusion = ROOT / "Models" / "CONCLUSION.md"
    if conclusion.exists():
        lines += ["", conclusion.read_text(encoding="utf-8").strip()]
    out = ROOT / "Models" / "RESULTS.md"
    out = ROOT / "Models" / "RESULTS.md"
    out.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"wrote {out} with {len(runs)} model(s): {', '.join(m['name'] for m, _ in runs.values())}")

write_results_md()

## Download (Google Colab only)
Colab deletes everything when the session ends. This saves the weights, results and figures as one zip.
Unzip it at the project root on your computer.

In [ ]:
if "google.colab" in sys.modules:
    import shutil
    from google.colab import files
    keep = [WEIGHTS, HISTORY, METRICS, ROOT / "Models" / "RESULTS.md", *FIGURES.glob("*.png")]
    pack = ROOT / "colab_download"
    for f in keep:
        dest = pack / f.relative_to(ROOT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, dest)
    files.download(shutil.make_archive(str(ROOT / f"{KIND}_results"), "zip", pack))